# 📈 સ્ટોક બ્રેકઆઉટ સ્કેનર - SL, T1, T2 સાથે

**મૂળ યુટ્યુબર સ્ટ્રેટેજી + સ્ટોપ લૉસ અને ટાર્ગેટ**

## આ કોડ શું કરે છે:
- ✅ 210 NSE સ્ટોક્સ ને સ્કેન કરે છે
- ✅ મૂવિંગ એવરેજ (30, 50, 200 દિવસ) ચેક કરે છે
- ✅ આપોઆપ સ્ટોપ લૉસ (SL) નિર્ધારણ કરે છે
- ✅ લક્ષ્ય ભાવ T1, T2 ગણતરી કરે છે
- ✅ એક્સેલ ફાઈલમાં પરિણામ સાચવે છે

## પગલું 1: જરૂરી લાઈબ્રેરી આયાત કરો

In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime
import warnings
import logging

logging.getLogger('yfinance').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

print('✅ બધી લાઈબ્રેરી તૈયાર છે!')

## પગલું 2: સ્કેનર ફંક્શન

In [ ]:
def advanced_stock_scanner(ticker_list):
    results = []
    today_date = datetime.now().strftime('%d-%m-%Y')
    
    print(f'🔍 {len(ticker_list)} સ્ટોક્સ સ્કેન કરી રહ્યા છીએ...\n')
    
    for ticker in ticker_list:
        try:
            data = yf.download(ticker, period='2y', interval='1d', progress=False)
            
            if data.empty or len(data) < 200:
                continue
            
            close_prices = data['Close'].squeeze()
            
            dma_30 = float(close_prices.rolling(window=30).mean().iloc[-1])
            dma_50 = float(close_prices.rolling(window=50).mean().iloc[-1])
            dma_200 = float(close_prices.rolling(window=200).mean().iloc[-1])
            cmp = float(close_prices.iloc[-1])
            
            dist_200_dma = ((cmp - dma_200) / dma_200) * 100
            
            last_1y_data = data.tail(252)
            high_date = last_1y_data['High'].squeeze().idxmax()
            car_data = close_prices.loc[high_date:]
            
            if len(car_data) < 10:
                continue
            
            car_values = car_data.expanding().mean()
            last_10_car = car_values.tail(10)
            
            car_status = 'Positive' if last_10_car.is_monotonic_increasing else 'Negative'
            
            if not ((cmp > dma_30) and (cmp > dma_50) and (cmp > dma_200) and (car_status == 'Positive')):
                continue
            
            swing_low = float(data['Low'].tail(10).min().item())
            sl = swing_low
            entry = cmp
            risk = entry - sl
            
            if risk <= 0:
                continue
            
            t1 = entry + (2 * risk)
            t2 = entry + (3 * risk)
            t3 = entry + (5 * risk)
            
            reward_percent = ((t2 - entry) / entry) * 100
            sl_percent = ((entry - sl) / entry) * 100
            rr_ratio = reward_percent / sl_percent if sl_percent > 0 else 0
            
            high_52 = float(data['High'].tail(252).max().item())
            dist_52 = ((high_52 - entry) / high_52) * 100
            
            returns = 0
            if len(close_prices) > 21:
                close_21_days_ago = float(close_prices.iloc[-21])
                returns = ((entry - close_21_days_ago) / close_21_days_ago) * 100
            
            if dma_30 > dma_50 > dma_200:
                trend = 'મજબૂત ઉપવતી ટ્રેન્ડ'
            else:
                trend = 'નબળો'
            
            results.append({
                'તારીખ': today_date,
                'સ્ટોક': ticker.replace('.NS', ''),
                'વર્તમાન_ભાવ': round(entry, 2),
                'SL': round(sl, 2),
                'SL_%': round(sl_percent, 2),
                'T1': round(t1, 2),
                'T2': round(t2, 2),
                'T3': round(t3, 2),
                'રીસ્ક': round(risk, 2),
                'લાભ_%': round(reward_percent, 2),
                'R_R_રેશિયો': round(rr_ratio, 2),
                'ટ્રેન્ડ': trend,
                'CAR': car_status
            })
        except:
            pass
    
    if results:
        df = pd.DataFrame(results)
        return df.sort_values(by='લાભ_%', ascending=False)
    else:
        return pd.DataFrame()

## પગલું 3: સ્કેનર ચલાવો

In [ ]:
my_stocks = [
    'PNB.NS', 'RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'ICICIBANK.NS',
    'HDFCBANK.NS', 'ASIANPAINT.NS', 'MARUTI.NS', 'WIPRO.NS', 'BAJAJFINSV.NS',
    'KOTAKBANK.NS', 'SBIN.NS', 'CIPLA.NS', 'SUNPHARMA.NS', 'TATASTEEL.NS'
]

print('\n' + '='*100)
print('🟢 સ્ટોક સ્કેનર - SL, T1, T2 સાથે')
print('='*100 + '\n')

result = advanced_stock_scanner(my_stocks)

## પગલું 4: પરિણામો દર્શાવો

In [ ]:
if result.empty:
    print('❌ આજે કોઈ સ્ટોક મળ્યો નહીં.')
else:
    print(result.to_string(index=False))
    print('\n' + '='*100)
    print(f'✅ કુલ સ્ટોક્સ: {len(result)}')
    print(f'📊 સરેરાશ SL %: {result["SL_%"].mean():.2f}%')
    print(f'📊 સરેરાશ લાભ: {result["લાભ_%"].mean():.2f}%')
    print(f'📊 સરેરાશ R:R: {result["R_R_રેશિયો"].mean():.2f}:1')

## પગલું 5: એક્સેલમાં સાચવો

In [ ]:
if not result.empty:
    result.to_excel('સ્ટોક_સ્કેનર_પરિણામ.xlsx', index=False)
    print('✅ પરિણામ સાચવ્યું: સ્ટોક_સ્કેનર_પરિણામ.xlsx')
else:
    print('❌ સાચવવા માટે ડેટા નથી.')